# Monte Carlo Methods

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caam37830-sci-comp/notes/blob/main/07_probability_statistics/monte_carlo.ipynb)

Monte Carlo methods use randomness to solve deterministic problems.  Most of them compute an integral written as an expectation,

$$ \mathbb E[f(X)] = \int f(x)\, p(x)\, dx, $$

where $X$ is a random variable with probability density $p$, a non-negative function that integrates to 1.  If $X_1, X_2, \dots$ are independent samples of $X$, the **law of large numbers** says that their sample mean converges to the expectation:

$$ \hat f_n = \frac1n \sum_{i=1}^n f(X_i) \;\to\; \mathbb E[f(X)] \quad\text{as } n \to \infty. $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.integrate import quad

rng = np.random.default_rng(37830)

## Example: estimating $\pi$

The unit disk $D$ has area $\pi$ and lies inside the square $[-1, 1]^2$, which has area 4.  If $U$ is uniformly distributed on the square, then

$$ \pi = 4\, P(U \in D) = 4\, \mathbb E[\chi_D(U)], $$

where the indicator function $\chi_D$ is 1 inside the disk and 0 outside.  So $\pi$ is about 4 times the fraction of random points in the square that land in the disk.

In [ ]:
n = 1000
U = rng.uniform(-1, 1, size=(n, 2))
inside = np.sum(U**2, axis=1) < 1

plt.figure(figsize=(5, 5))
plt.scatter(U[~inside, 0], U[~inside, 1], s=4)
plt.scatter(U[inside, 0], U[inside, 1], s=4)
plt.gca().set_aspect('equal')
plt.show()

estimate = 4 * np.mean(inside)
print(f"estimate {estimate}, error {estimate - np.pi:+.4f}")

## The error of Monte Carlo

The estimate $\hat f_n$ is random, and so is its error.  Since the $X_i$ are independent, the variance of their sum is the sum of their variances:

$$ \mathrm{Var}(\hat f_n) = \frac{1}{n^2} \sum_{i=1}^n \mathrm{Var}(f(X_i)) = \frac{\sigma^2}{n}, \qquad \sigma^2 = \mathrm{Var}(f(X)). $$

The typical (root-mean-square) error is therefore $\sigma/\sqrt n$.  The **central limit theorem** says for large $n$, $\hat f_n$ is approximately normally distributed, with mean $\mathbb E[f(X)]$ and standard deviation $\sigma/\sqrt n$.

The rate $n^{-1/2}$ is slow, since each additional correct digit costs 100 times as many samples.  But it doesn't depend on the dimension of $x$ or on the smoothness of $f$.  A quadrature rule of order $p$ that uses $n$ points in $d$ dimensions has an error of $O(n^{-p/d})$, so Monte Carlo is more accurate when $d > 2p$, and integrals in many dimensions are almost always computed by Monte Carlo.

We check the central limit theorem by repeating the $\pi$ experiment 2000 times.  Each sample $4\chi_D(U)$ is 4 with probability $\pi/4$ and 0 otherwise, so $\sigma^2 = 16 \cdot \frac{\pi}{4}\left(1 - \frac{\pi}{4}\right)$.

In [ ]:
n, repeats = 1000, 2000
hits = np.sum(rng.uniform(-1, 1, size=(repeats, n, 2))**2, axis=2) < 1   # (repeats, n)
estimates = 4 * np.mean(hits, axis=1)
sigma = 4 * np.sqrt(np.pi / 4 * (1 - np.pi / 4))

bins = 4 * (np.arange(730, 842, 3) + 0.5) / n      # estimates are multiples of 4/n
x = np.linspace(2.9, 3.4, 300)
plt.hist(estimates, bins=bins, density=True, alpha=0.5, label=f'{repeats} estimates with $n = {n}$')
plt.plot(x, stats.norm.pdf(x, loc=np.pi, scale=sigma / np.sqrt(n)), 'k',
         label=r'normal, mean $\pi$, std. $\sigma/\sqrt{n}$')
plt.legend(loc='upper left')
plt.show()

Finally, let's look at how the error converges with $n$:

In [ ]:
ns = np.int32(np.logspace(0,6,100))
errs = []
for n in ns:
    xs = 2*np.random.rand(2,n)-1
    ids = np.linalg.norm(xs,axis=0)<1
    errs.append((4*sum(ids)/n)-np.pi)
    
plt.loglog(ns,np.abs(errs),'o-')
plt.loglog(ns,1/np.sqrt(ns),'k--')
plt.xlabel('n')
plt.ylabel('Error')
plt.show()

### Error bars

We usually don't know $\sigma$, but we can estimate it from the same samples by the sample standard deviation $s$.  The **standard error** $s/\sqrt n$ then estimates the error, and by the central limit theorem

$$ \hat f_n \pm 1.96\, \frac{s}{\sqrt n} $$

is an approximate 95% confidence interval (see [SciPy `stats`](scipy_stats.ipynb)).  A Monte Carlo estimate should always be reported with an error bar like this, and it costs nothing extra to compute.

In [ ]:
def mc_estimate(fx):
    """
    sample mean and standard error of the array of samples fx
    """
    return np.mean(fx), np.std(fx, ddof=1) / np.sqrt(len(fx))


for m in [10**2, 10**4, 10**6]:
    U = rng.uniform(-1, 1, size=(m, 2))
    est, se = mc_estimate(4 * (np.sum(U**2, axis=1) < 1))
    print(f"n = {m:>7d}:  pi = {est:.5f} +/- {1.96 * se:.5f},   actual error {est - np.pi:+.5f}")

About 95% of these intervals should contain $\pi$.  We check with the 2000 repetitions above:

In [ ]:
se = 4 * np.std(hits, axis=1, ddof=1) / np.sqrt(n)
covered = np.abs(estimates - np.pi) < 1.96 * se
print(f"{np.mean(covered):.1%} of the intervals contain pi")

The error bar is only as good as the estimate $s$ of $\sigma$.  It can be badly wrong when $f(X)$ is very large with a small probability, as in an example below.

## Importance sampling

Suppose we want the integral of a sharply peaked function,

$$ I = \int_{-\infty}^\infty e^{-10^4 x^2}\, dx = \frac{\sqrt\pi}{100}. $$

The integrand is negligible outside $[-1, 1]$, so we could write $I = 2\,\mathbb E[f(U)]$ with $U$ uniform on $[-1, 1]$.  But the peak has a width of about $0.01$, so almost every sample misses it, and $f(U)$ has a large variance compared with its mean.

Instead, draw samples from a density $q$ that puts more of them where $f$ is large, and correct for this with a weight:

$$ I = \int \frac{f(x)}{q(x)}\, q(x)\, dx = \mathbb E\!\left[\frac{f(Y)}{q(Y)}\right], \qquad Y \sim q. $$

This is **importance sampling**.  The estimate is unbiased for any density $q$ that is positive wherever $f$ is nonzero, but its variance depends on $q$.  We try normal distributions $q$ with several standard deviations $s$, and measure the root-mean-square error over 50 repetitions.

In [ ]:
f = lambda x: np.exp(-1e4*x*x)
xs = np.linspace(-1,1,1000)
plt.plot(xs,f(xs))
plt.show()

In [ ]:
exact = np.sqrt(np.pi) / 100
ns = 2 ** np.arange(4, 17)
trials = 50


def rms_relative_error(sample, weight):
    """
    RMS relative error of the estimate mean(weight(Y)) over the trials, for each n in ns

    sample(size) returns an array of samples Y with the given shape
    """
    errors = []
    for n in ns:
        est = np.mean(weight(sample((trials, n))), axis=1)
        errors.append(np.sqrt(np.mean((est - exact)**2)) / exact)
    return np.array(errors)


plt.figure(figsize=(7, 4.5))
err = rms_relative_error(lambda size: rng.uniform(-1, 1, size), lambda y: 2 * f(y))
plt.loglog(ns, err, 'k*--', label='uniform on $[-1, 1]$')

for s in [1, 0.1, 0.01, 0.005]:
    q = stats.norm(scale=s)
    err = rms_relative_error(lambda size: q.rvs(size=size, random_state=rng), lambda y: f(y) / q.pdf(y))
    plt.loglog(ns, err, 'o-', label=f'normal, $s = {s}$')

plt.loglog(ns, 1 / np.sqrt(ns), 'k:', label=r'$n^{-1/2}$')
plt.xlabel('$n$')
plt.ylabel('RMS relative error')
plt.legend()
plt.show()

The best of these proposals is $s = 0.01$.  Its error is more than 20 times smaller than that of uniform sampling.  The choice $s = 1/\sqrt{2 \cdot 10^4} \approx 0.007$ would be even better.  Of course, normalizing $q$ requires knowing $I$, but we should aim for $q$ to have roughly the shape of $|f|$.



## Markov chain Monte Carlo

Often we want $\mathbb E[f(X)]$ for a density we can't sample from directly.  A common case, in statistical physics and in Bayesian statistics, is

$$ p(x) = \frac{e^{-V(x)}}{Z}, \qquad Z = \int e^{-V(x)}\, dx, $$

where we can evaluate $V$, but the normalizing constant $Z$ is unknown, because it is itself a hard integral in high dimensions.

**Markov chain Monte Carlo (MCMC)** generates a sequence $X_0, X_1, X_2, \dots$ in which each $X_{i+1}$ is drawn at random given only $X_i$.  Such a sequence is a Markov chain.  We design the chain so that its stationary distribution is $p$: after many steps the $X_i$ are distributed according to $p$, and averages along the chain converge to $\mathbb E[f(X)]$.

### The Metropolis algorithm

Starting from $X_i$:

1. Propose a move $Y = X_i + \delta\, \xi$, where $\xi \sim \mathcal N(0, 1)$ and $\delta$ is a step size.
2. Accept it, $X_{i+1} = Y$, with probability $\min\left(1, p(Y)/p(X_i)\right)$.  Otherwise reject it and stay put: $X_{i+1} = X_i$.

Moves to higher probability are always accepted, and moves to lower probability only sometimes.  The ratio $p(Y)/p(X_i) = e^{V(X_i) - V(Y)}$ doesn't involve $Z$.  This is a specialization of the [Metropolis-Hastings algorithm](https://en.wikipedia.org/wiki/Metropolis%E2%80%93Hastings_algorithm), you can check the stationary probability is $p$.

In [ ]:
def metropolis(V, x0, delta, nsteps, rng):
    """
    Metropolis chain for the density proportional to exp(-V(x)), started at x0

    returns the chain X_0, ..., X_nsteps and the fraction of proposals accepted
    """
    X = np.empty(nsteps + 1)
    X[0] = x0
    xi = rng.standard_normal(nsteps)        # draw the random numbers in bulk
    u = rng.random(nsteps)
    accepted = 0
    for i in range(nsteps):
        y = X[i] + delta * xi[i]
        if u[i] < np.exp(V(X[i]) - V(y)):   # same as u < min(1, ratio), since u < 1
            X[i + 1] = y
            accepted += 1
        else:
            X[i + 1] = X[i]
    return X, accepted / nsteps

We sample from $p(x) \propto e^{-V(x)}$ with $V(x) = x^4 - x^2$, which has two shallow wells at $x = \pm 1/\sqrt 2$, and compare with the exact density, normalized by quadrature.

In [ ]:
V = lambda x: x**4 - x**2
Z = quad(lambda x: np.exp(-V(x)), -np.inf, np.inf)[0]

X, acceptance = metropolis(V, x0=0.0, delta=1.0, nsteps=100_000, rng=rng)
X = X[1000:]              # discard a burn-in period, see below

x = np.linspace(-2, 2, 300)
plt.hist(X, bins=60, density=True, alpha=0.5, label='Metropolis samples')
plt.plot(x, np.exp(-V(x)) / Z, 'k', label=r'$e^{-V(x)}/Z$')
plt.legend()
plt.show()

exact_x2 = quad(lambda x: x**2 * np.exp(-V(x)), -np.inf, np.inf)[0] / Z
print(f"acceptance rate {acceptance:.2f}")
print(f"E[X^2]: MCMC {np.mean(X**2):.4f}, quadrature {exact_x2:.4f}")

### Choosing the step size

Successive states of a Markov chain are correlated, and the step size $\delta$ controls how strongly.  Trace plots of chains started far from the wells, at $x_0 = 3$, show the trade-off.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for a, delta in zip(ax, [0.05, 1.0, 20.0]):
    chain, acceptance = metropolis(V, x0=3.0, delta=delta, nsteps=2000, rng=rng)
    a.plot(chain, lw=0.7)
    a.set_ylabel('$X_i$')
    a.set_title(rf'$\delta = {delta}$, acceptance rate {acceptance:.2f}')
ax[-1].set_xlabel('step $i$')
plt.tight_layout()
plt.show()

* With a small step, almost every proposal is accepted, but the chain moves like a random walk with tiny steps.  It needs many steps just to reach the wells from $x_0 = 3$, and it rarely moves from one well to the other.
* With a large step, almost every proposal lands where $p$ is tiny and is rejected, so the chain is stuck for long stretches.
* A moderate step explores the distribution quickly.

The first part of a chain, before it has forgotten its starting point, doesn't sample $p$ and should be discarded as **burn-in**.

## Exercises

### Exercise 1

The unit ball in $d$ dimensions has volume $\pi^{d/2} / \Gamma(d/2 + 1)$ (`scipy.special.gamma`).  Estimate it as $2^d$ times the fraction of $n = 10^6$ uniform points in $[-1, 1]^d$ that land in the ball, for $d = 2, 4, \dots, 16$.  Report each estimate with an error bar, and plot the relative error against $d$.  Why does the relative error grow with $d$, even though the rate $n^{-1/2}$ doesn't depend on the dimension?

In [ ]:
# Your code here